![](https://github.com/datagong/data/blob/main/datagong.png?raw=true)

© DATAGONG - Tous droits réservés - 2026

📋 **Rappels Conditions Générales d'Utilisation**

⚠️ Les Notebooks sont privés

❌ partage des Notebooks, de leur contenu, des liens, des images, …

❌ publier les Notebooks sur GitHub (public) ou tout autre outil de versionning

✅ sauvegarder les Notebooks dans votre environnement personnel et privé

✅ réutiliser les codes dans le cadre de vos projets en entreprise / projets personnels / etc.

✅ annoter / modifier les Notebooks dans votre environnement personnel et privé

# <center><u><b>Data Visualization avec Streamlit et Plotly</b></u></center>

# <b>Streamlit + Plotly — 4. Interactivité : Widgets & Session State</b>

Dans le notebook précédent, nous avons construit un **design system** pour harmoniser nos graphiques Plotly. Nos visualisations sont propres et cohérentes, mais pour l'instant elles restent **statiques** : l'utilisateur ne peut pas choisir ce qu'il veut voir. C'est exactement ce que nous allons changer ici.

Streamlit propose des composants d'interface appelés [**widgets**](https://docs.streamlit.io/develop/api-reference/widgets) : des menus déroulants, des sliders, des sélecteurs de dates, etc. Ces widgets permettent à l'utilisateur de **filtrer, trier ou explorer** les données affichées, directement depuis le navigateur, sans toucher au code. Concrètement, vous allez créer des filtres que vos utilisateurs pourront manipuler pour personnaliser les graphiques en temps réel.

Mais il y a un piège : Streamlit **ré-exécute l'intégralité du script** à chaque interaction. Si vous ne faites rien de particulier, les choix de l'utilisateur sont perdus à chaque clic. C'est là qu'intervient [`st.session_state`](https://docs.streamlit.io/develop/api-reference/caching-and-state/st.session_state), un dictionnaire persistant qui permet de **mémoriser l'état de l'interface** entre deux exécutions. Nous verrons aussi comment utiliser des **callbacks** pour déclencher des actions automatiques dès qu'un widget change de valeur.

À la fin de ce notebook, votre application passera d'un affichage figé à un véritable outil d'exploration interactif.

# 0. Créer un panneau latéral de filtres

Dans Streamlit, la **sidebar** est un panneau latéral rétractable, idéal pour y regrouper des filtres sans encombrer la zone principale. Vous l'activez avec le bloc `with st.sidebar:` — tout ce que vous placez à l'intérieur apparaîtra dans ce panneau.

Pour nos filtres, nous allons utiliser deux widgets clés :

[`st.multiselect(label, options, default)`](https://docs.streamlit.io/develop/api-reference/widgets/st.multiselect) affiche un menu déroulant dans lequel l'utilisateur peut cocher **plusieurs valeurs** à la fois. Le paramètre `options` contient la liste des choix possibles, et `default` définit la sélection initiale. Dans notre cas, nous l'utiliserons pour filtrer par catégorie de produit.

[`st.date_input(label, value)`](https://docs.streamlit.io/develop/api-reference/widgets/st.date_input) affiche un petit calendrier pour choisir une date. Le paramètre `value` fixe la date affichée par défaut. Nous en placerons deux — un pour la date de début, un pour la date de fin — afin de définir une période de filtrage.

Une fois ces widgets en place, il faut **sauvegarder** les choix de l'utilisateur pour qu'ils soient accessibles ailleurs dans l'application. C'est le rôle de [`st.session_state`](https://docs.streamlit.io/develop/api-reference/caching-and-state/st.session_state) : un dictionnaire partagé dans toute l'application. Ici, nous utilisons une simple affectation `st.session_state["clé"] = valeur` pour mettre à jour l'état à chaque interaction.

⚠️ Attention à ne pas utiliser `setdefault()` dans ce cas : cette méthode n'écrit que si la clé n'existe pas encore, ce qui empêcherait la mise à jour quand l'utilisateur modifie un filtre.

Voyons cela en pratique :

In [1]:
%%writefile -a ../streamlit_app/app.py
# --- Section: Filtres (sidebar) ---
import datetime as dt

with st.sidebar:
    st.header("🎛️ Filtres")  # Titre du panneau latéral
    cats = sorted(list(data["categorie"].cat.categories))  # Liste triée des catégories
    f_cats = st.multiselect("Catégories", options=cats, default=cats)  # Sélection multiple des catégories
    dmin = st.date_input("Date min", value=data["date"].min().date())  # Sélection de la date minimale
    dmax = st.date_input("Date max", value=data["date"].max().date())  # Sélection de la date maximale

# État partagé — on met à jour session_state à chaque exécution
st.session_state["f_cats"] = f_cats  # Sauvegarde des catégories sélectionnées
st.session_state["dmin"] = dmin      # Sauvegarde de la date min sélectionnée
st.session_state["dmax"] = dmax      # Sauvegarde de la date max sélectionnée

Appending to ../streamlit_app/app.py


Votre sidebar est en place. L'utilisateur peut désormais filtrer les données par catégorie et par période, et ces choix sont mémorisés dans `st.session_state`. Il ne reste plus qu'à **connecter ces filtres aux graphiques** pour que l'affichage se mette à jour automatiquement.

💡 Si vous relancez l'application (`streamlit run app.py`), vous verrez le panneau latéral apparaître à gauche avec vos filtres. Pour l'instant, ils ne modifient rien — c'est ce que nous allons faire juste après.

# 1. Appliquer les filtres et mettre à jour les graphiques

## 1.1. Comment les filtres interagissent avec les graphiques

Le principe est simple : à chaque fois que l'utilisateur modifie un filtre, nous récupérons ses choix depuis `st.session_state`, nous filtrons le DataFrame en conséquence, puis nous regénérons le graphique avec les données filtrées. Comme Streamlit ré-exécute le script à chaque interaction, la mise à jour est **automatique** — vous n'avez rien de plus à coder pour rafraîchir l'affichage.

Dans le code ci-dessous, la fonction `filter_data` prend en entrée les catégories sélectionnées, la date min et la date max, et renvoie un DataFrame ne contenant que les lignes correspondantes. Ce DataFrame filtré est ensuite passé à `make_line` pour dessiner le graphique.

## 1.2. Donner un retour visuel avec [`st.toast`](https://docs.streamlit.io/develop/api-reference/status/st.toast)

[`st.toast(message, icon)`](https://docs.streamlit.io/develop/api-reference/status/st.toast) affiche une petite notification temporaire en bas à droite de l'application. C'est un retour discret mais utile : ici, nous l'utilisons pour afficher le nombre de lignes restantes après filtrage (par exemple : *"1 234 lignes après filtre"*). L'utilisateur voit immédiatement l'impact de ses choix sans que cela ne perturbe sa navigation.

In [2]:
%%writefile -a ../streamlit_app/app.py
# --- Section: Application des filtres ---

# 1. Construire le dictionnaire de filtres depuis les variables des widgets
filtres = dict(
    categorie=f_cats,
    date_min=dmin,
    date_max=dmax
)

# 2. Appliquer les filtres au DataFrame
df_filtered = filter_data(data, **filtres)

# 3. Afficher un message contextuel avec le nombre de lignes restantes
st.toast(f"{len(df_filtered):,} lignes après filtre".replace(",", " "), icon="✅")

# 4. Générer et afficher le graphique filtré
fig_filt = make_line(
    df_filtered, x="date", y="ventes", color="categorie",
    title="Ventes filtrées"
)
st.plotly_chart(fig_filt, use_container_width=True)

Appending to ../streamlit_app/app.py


# <font color='#ff7373'><b>Félicitations !</b></font>

Vous venez de rendre votre application **interactive** : vos utilisateurs peuvent désormais filtrer les données par catégorie et par période, et voir les graphiques se mettre à jour en temps réel. C'est une étape clé — vous avez transformé un tableau de bord statique en un véritable outil d'exploration.

Dans le prochain notebook, nous structurerons l'application en **plusieurs pages** et travaillerons la mise en page pour un rendu plus professionnel.

# <center><font color='#3b4859'><u>![](https://github.com/datagong/data/blob/main/mini%20datagong%202.png?raw=true)</u></font></center>